# YouTube Video Notes Generator

## Turn Transcripts into Structured Study Notes - Prompt Engineering in Practice

**Project 21 / 100** -- GenAI Learning Series


## Project Overview

The YouTube Notes Generator transforms any video transcript into five complementary study formats:
- Structured Notes
- Executive Summary  
- Key Takeaways
- Action Items
- Quiz Questions

Learn prompt engineering patterns: role specification, output formatting, few-shot examples, and validation.


## Learning Goals

1. Parse and preprocess raw transcripts
2. Prompt engineering patterns for reliable extraction
3. Multi-stage pipeline design
4. Quality evaluation without ground truth labels
5. Fault-tolerant system design


## Problem Statement

Students watch 60-90 minute YouTube videos but can't quickly extract:
- Study notes by topic
- Executive summary
- Key takeaways for review
- Action items to implement
- Self-test questions

Goal: Auto-generate five complementary study formats from raw transcript.


## Why This Matters

Educational content is enormous. Auto-generated study materials save students 5-10 hours per video course.
Real market opportunity for EdTech platforms.


## Transcript Corpus

In [ ]:
TRANSCRIPTS = [
    {
        "video_id": "ML-001",
        "title": "Introduction to Neural Networks",
        "channel": "AI Learning Hub",
        "duration_minutes": 45,
        "transcript": "Hello everyone! Today we discuss neural networks - fundamental to deep learning. A neural network is a mathematical model inspired by the biological brain with layers of interconnected nodes. Each neuron receives inputs, applies transformation, and passes to next layer. Key insight: networks learn complex non-linear functions from data. Let me break down components. Input layer takes raw features. Hidden layers learn abstract representations. Output layer produces predictions. Neurons compute weighted sum of inputs, add bias, apply non-linear activation. Common activations: ReLU, sigmoid, tanh. ReLU is most popular. Training involves forward pass through network and backward pass with backpropagation. Update weights using optimizers like SGD or Adam. Learning rate controls step size. Too high diverges, too low is slow. Avoid overfitting with dropout and regularization. Thanks for watching!"
    },
    {
        "video_id": "PROD-001",
        "title": "5 Productivity Hacks",
        "channel": "Productivity Ninja",
        "duration_minutes": 22,
        "transcript": "Hey everyone! Five productivity hacks. First: time blocking. Divide day into time blocks. Reduces context switching which costs 15-20 minutes. Second: two-minute rule. Tasks under two minutes do immediately. Dont add to todo. Third: Pomodoro technique. Work 25 minutes, break 5 minutes. After four cycles take longer break. Your brain has natural focus cycles. Fourth: batch similar tasks. Dont check email all day. Check three times daily. Reduces decision fatigue. Fifth: plan tomorrow tonight. Write three most important tasks before bed. Your mind works on them while sleeping. Bonus: turn off notifications. They are context switching bombs. Pick one hack, master it two weeks, then add another. Habit stacking works. Thanks for watching!"
    },
    {
        "video_id": "DATA-001",
        "title": "Data Pipelines at Scale",
        "channel": "Data Engineering Today",
        "duration_minutes": 55,
        "transcript": "Hey, Alex here from Data Engineering Today. Today data pipelines at scale. A pipeline ingests raw data, transforms it, loads to destination - ETL. Small scale single cron job works. At scale gets complex. Need fault tolerance, monitoring, idempotency, cost optimization. Fault tolerance first. What if pipeline fails? Must design with failure in mind. Idempotency important: running twice produces same result. Track processed data, skip on retry. Monitoring essential. Failures cant be silent. Set alerts on latency, data quality, failure rates. Cost often overlooked. Pipeline might work but be inefficient. Reading terabytes when gigabytes needed. At scale optimizing saves millions yearly. Batch vs stream. Batch schedule based, easier to reason about. Streams run continuously, lower latency but harder to debug. Tools: Spark, dbt for batch. Kafka, Flink for streams. Data lives where? Warehouses like Snowflake great for analytics. Lakes cheaper, need governance. Lakehouses merge both. Thanks for watching!"
    },
]

print(f"Loaded {len(TRANSCRIPTS)} transcripts")
for t in TRANSCRIPTS:
    words = len(t["transcript"].split())
    print(f"  [{t['video_id']}] {words} words")


## Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import nltk
    nltk.data.find("tokenizers/punkt")
except LookupError:
    import nltk
    nltk.download("punkt", quiet=True)

print("Environment ready.")


## Imports

In [ ]:
import re, json, random, textwrap, warnings
from dataclasses import dataclass
from typing import List, Dict, Tuple

import nltk
from nltk.tokenize import sent_tokenize

warnings.filterwarnings("ignore")

random.seed(42)

MAX_NOTES_ITEMS = 10
MAX_TAKEAWAYS = 8
MAX_ACTION_ITEMS = 5
MAX_QUIZ_QUESTIONS = 5

print("All imports loaded.")


## Prompt Engineering Principles

| Principle | Example |
|-----------|---------|
| Clear role | "You are a technical note-taker" |
| Specify format | "Output JSON with keys: title, bullets" |
| Use examples | Show 1-2 example outputs |
| Constraints | "Max 3 bullets per topic" |
| Context | Provide domain background |
| Validation | Check output matches expected format |

Demonstrate patterns using template-based extraction and NLP.


## Transcript Preprocessing

In [ ]:
@dataclass
class TranscriptSegment:
    timestamp_start: str
    timestamp_end: str
    text: str
    segment_id: int

def preprocess_transcript(transcript_text: str, title: str) -> Tuple[str, List[TranscriptSegment]]:
    text = re.sub(r'\s+', ' ', transcript_text).strip()
    sentences = sent_tokenize(text)
    
    segments = []
    chunk_size = 4
    for i in range(0, len(sentences), chunk_size):
        chunk = ' '.join(sentences[i:i+chunk_size])
        start_min = int((i / len(sentences)) * 45) if sentences else 0
        end_min = int(((i + chunk_size) / len(sentences)) * 45) if sentences else 0
        
        segments.append(TranscriptSegment(
            timestamp_start=f"{start_min:02d}:{random.randint(0, 59):02d}",
            timestamp_end=f"{end_min:02d}:{random.randint(0, 59):02d}",
            text=chunk,
            segment_id=i // chunk_size,
        ))
    
    return text, segments


sample_tx = TRANSCRIPTS[0]
full_text, segments = preprocess_transcript(sample_tx["transcript"], sample_tx["title"])
print(f"Sentences: {len(sent_tokenize(full_text))}, Segments: {len(segments)}")


## Extraction Functions

In [ ]:
def extract_notes(full_text: str, segments: List[TranscriptSegment]) -> Dict:
    sentences = sent_tokenize(full_text)
    scored = [(s, len(s.split())) for s in sentences if 8 <= len(s.split()) <= 25]
    scored.sort(key=lambda x: x[1], reverse=True)
    main_points = [s[0] for s in scored[:MAX_NOTES_ITEMS]]
    
    topics = []
    for i, seg in enumerate(segments[::2]):
        heading = ' '.join(seg.text.split()[:5])
        bullets = main_points[i*3:(i+1)*3] if i*3 < len(main_points) else ['...']
        topics.append({"heading": heading, "bullets": bullets, "timestamp": seg.timestamp_start})
    
    return {"title": "Study Notes", "topics": topics[:5]}


def generate_summary(full_text: str) -> str:
    sentences = sent_tokenize(full_text)
    filtered = [s for s in sentences if 10 <= len(s.split()) <= 30]
    if not filtered:
        filtered = sentences
    
    n = len(filtered)
    indices = [0, n//2 if n > 1 else 0, n-1 if n > 1 else 0]
    summary_sents = [filtered[i] for i in indices if i >= 0 and i < len(filtered)]
    return ' '.join(summary_sents)


def extract_takeaways(full_text: str) -> List[str]:
    sentences = sent_tokenize(full_text)
    verbs = ["use", "learn", "remember", "avoid", "understand", "focus"]
    takeaways = []
    for sent in sentences:
        if any(v in sent.lower() for v in verbs) and 8 <= len(sent.split()) <= 20:
            takeaways.append(sent.strip())
    
    return list(dict.fromkeys(takeaways))[:MAX_TAKEAWAYS]


notes = extract_notes(full_text, segments)
summary = generate_summary(full_text)
takeaways = extract_takeaways(full_text)

print(f"Notes: {len(notes['topics'])} topics")
print(f"Summary: {summary[:80]}...")
print(f"Takeaways: {len(takeaways)} items")


## Action Items and Quiz

In [ ]:
def extract_actions(full_text: str) -> List[Dict]:
    sentences = sent_tokenize(full_text)
    imperatives = []
    for sent in sentences:
        words = sent.split()
        if words and words[0].lower() in ["use", "add", "set", "enable", "implement", "track"]:
            imperatives.append(sent.strip())
    
    actions = []
    for i, sent in enumerate(imperatives[:MAX_ACTION_ITEMS]):
        priorities = ["High", "Medium", "Low"]
        actions.append({"priority": priorities[i % 3], "action": sent, "time": "15-30 min"})
    
    return actions


def generate_quiz(full_text: str) -> List[Dict]:
    sentences = sent_tokenize(full_text)
    selected = [s for s in sentences if 15 <= len(s.split()) <= 30][:MAX_QUIZ_QUESTIONS]
    
    questions = []
    for sent in selected:
        words = [w.strip(",.!?;") for w in sent.split()]
        key_word = next((w for w in words if len(w) > 4), "concept")
        questions.append({
            "question": f"What is {key_word}?",
            "options": ["A) " + sent[:25], "B) Related", "C) Wrong", "D) False"],
            "correct": "A",
        })
    
    return questions


actions = extract_actions(full_text)
quiz = generate_quiz(full_text)
print(f"Actions: {len(actions)}, Quiz: {len(quiz)}")


## Video Notes Generator

In [ ]:
class VideoNotesGenerator:
    def __init__(self, video: Dict):
        self.video = video
        self.full_text, self.segments = preprocess_transcript(video["transcript"], video["title"])
    
    def generate_all(self) -> Dict:
        return {
            "video": {"title": self.video["title"], "channel": self.video["channel"]},
            "notes": extract_notes(self.full_text, self.segments),
            "summary": generate_summary(self.full_text),
            "takeaways": extract_takeaways(self.full_text),
            "action_items": extract_actions(self.full_text),
            "quiz": generate_quiz(self.full_text),
        }
    
    def to_markdown(self, data: Dict) -> str:
        md = f"# {data['video']['title']}\n\n"
        md += f"**Channel:** {data['video']['channel']}\n\n"
        md += f"## Summary\n\n{data['summary']}\n\n"
        md += "## Key Takeaways\n\n"
        for i, t in enumerate(data['takeaways'], 1):
            md += f"{i}. {t}\n"
        md += "\n## Action Items\n\n"
        for a in data['action_items']:
            md += f"- [{a['priority']}] {a['action']}\n"
        return md


gen = VideoNotesGenerator(TRANSCRIPTS[0])
output = gen.generate_all()
print("Generator ready - processing...")
print(gen.to_markdown(output)[:400])


## Batch Processing

In [ ]:
results = []
for video in TRANSCRIPTS:
    print(f"Processing {video['title']}...", end=" ")
    gen = VideoNotesGenerator(video)
    results.append(gen.generate_all())
    print("OK")

print(f"\nProcessed {len(results)} videos")


## Quality Evaluation

In [ ]:
def evaluate(results: List[Dict]) -> Dict:
    metrics = {"topics": [], "summary_len": [], "takeaways": [], "actions": [], "quiz": []}
    
    for res in results:
        metrics["topics"].append(len(res["notes"].get("topics", [])))
        metrics["summary_len"].append(len(res["summary"].split()))
        metrics["takeaways"].append(len(res["takeaways"]))
        metrics["actions"].append(len(res["action_items"]))
        metrics["quiz"].append(len(res["quiz"]))
    
    return {k: (sum(v)/len(v) if v else 0) for k, v in metrics.items()}


quality = evaluate(results)
print("Quality Metrics:")
for k, v in quality.items():
    print(f"  {k}: {v:.1f}")


## Sample Output

In [ ]:
if results:
    print(VideoNotesGenerator(TRANSCRIPTS[0]).to_markdown(results[0]))


## Prompt Engineering Lessons

1. Be specific - "Extract 5 bullets" beats "Extract points"
2. Show examples - Few-shot beats explanation
3. Define format - Specify exact JSON or markdown style
4. Role-play helps - "You are a technical writer"
5. Add constraints - "Max 3 bullets per topic"
6. Validate output - Always check format
7. Use domain vocabulary - Technical terms matter


## Limitations

- Uses heuristics not LLM judgment
- Generic extraction, no domain tuning
- No factual verification
- Mock timestamps not real video sync
- Single-speaker assumption


## Common Mistakes

| Mistake | Fix |
|---------|-----|
| Vague prompts | Be specific |
| No examples | Add examples |
| Ambiguous format | Specify exact format |
| No validation | Always check output |
| Single attempt | Retry with variations |


## Mini Challenge

1. Add "Key Arguments" mode for debates
2. Improve scoring with TF-IDF
3. Add speaker diarization
4. Real YouTube transcript API
5. Multi-language support


## Production Considerations

Real transcripts via YouTube API or AssemblyAI. Cache results per video. Use LLM for accuracy. Support multiple export formats. Track user satisfaction.


## How to Improve

1. Replace heuristics with GPT-4/Claude
2. Fine-tune on domain examples
3. User role-specific generation
4. Feedback loops for tuning
5. Real-time streaming generation
6. Notion/OneNote integrations


## Key Takeaways

1. Prompt engineering is learnable - specificity, examples, constraints work
2. Structure matters - define formats precisely
3. One source, five outputs - modular design multiplies value
4. Heuristics often sufficient - don't always need LLMs
5. Evaluation without labels - use quality metrics + user feedback
6. Educational content valuable - real market opportunity
7. Keep pipelines modular - parameterize everything
